# 01 · Data Exploration — RideEat tickets
Team Paradox · TensorForge 2.0 Phase 2

Goal: understand label balance, languages, channels, multi-label structure, urgency, and adversarial patterns before modelling.

In [ ]:
import pandas as pd, numpy as np, re, unicodedata
import matplotlib.pyplot as plt
pd.set_option('display.max_colwidth', 200)
DATA = '../data/raw'
# keep_default_na=False so the text 'NA'/'null' inside tickets is not turned into NaN
read = lambda f: pd.read_csv(f'{DATA}/{f}', keep_default_na=False, na_values=[''], encoding='utf-8')
train, val = read('train.csv'), read('validation.csv')
print(train.shape, val.shape); train.head()

## 1. Integrity checks

In [ ]:
for name, d in [('train', train), ('val', val)]:
    print(name, '| unique ids:', d.ticket_id.is_unique,
          '| duplicate texts:', d.duplicated('text').sum(),
          '| empty text:', (d.text.str.strip() == '').sum())
print('train/val text overlap:', len(set(train.text) & set(val.text)))
print('is_urgent dtype:', train.is_urgent.dtype)

## 2. Category distribution (imbalance)

In [ ]:
ax = train.category.value_counts().plot.barh(figsize=(7,4), title='Primary category (train)')
ax.invert_yaxis(); plt.tight_layout(); plt.show()
train.category.value_counts(normalize=True).round(3)

## 3. Language × category

In [ ]:
pd.crosstab(train.category, train.language, margins=True)

In [ ]:
train.language.value_counts(normalize=True).round(3).plot.pie(autopct='%.0f%%', figsize=(4,4), ylabel='')
plt.title('Language share'); plt.show()

## 4. Channel and subject

In [ ]:
print(pd.crosstab(train.category, train.channel))
print('subject present by channel:'); print(train.subject.notna().groupby(train.channel).mean())

## 5. Text length by language

In [ ]:
train['n_chars'] = train.text.str.len()
train.boxplot('n_chars', by='language', figsize=(7,4)); plt.suptitle(''); plt.show()
train.groupby('language').n_chars.describe().round(0)

## 6. Secondary category — which pairs occur?

In [ ]:
print('share with secondary:', train.secondary_category.notna().mean())
train[train.secondary_category.notna()].groupby(['category','secondary_category']).size()

## 7. Urgency

In [ ]:
print('urgent share:', train.is_urgent.mean())
pd.crosstab(train.category, train.is_urgent)

In [ ]:
pd.crosstab(train.language, train.is_urgent, normalize='index').round(3)

## 8. Adversarial patterns
Prompt-injection lines and quoted/resolved correspondence (see DATA_NOTES).

In [ ]:
inj = train.text.str.contains(r'ignore|previous instructions|mark this|classify', case=False)
print('injection-like tickets:', inj.sum())
print(train[inj].category.value_counts())
train[inj][['language','category','is_urgent','text']].sample(5, random_state=0)

In [ ]:
quoted = train.text.str.contains(r'resolved|wrote:|^>', case=False, regex=True)
print('quoted/resolved-like tickets:', quoted.sum())
train[quoted][['channel','subject','category','text']].sample(3, random_state=0)

## 9. Read a few tickets per category

In [ ]:
for cat in sorted(train.category.unique()):
    print('='*20, cat)
    for _, r in train[train.category == cat].sample(2, random_state=1).iterrows():
        print(f'[{r.language}/{r.channel}] urgent={r.is_urgent} sec={r.secondary_category}')
        print('  ', r.text[:250].replace('\n', ' / '))

## 10. Validation vs train distribution

In [ ]:
pd.DataFrame({'train': train.category.value_counts(normalize=True),
              'val': val.category.value_counts(normalize=True)}).round(3)

## Findings (fill in)
- 
- 
- 